In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import statsmodels.formula.api as smf


In [2]:
df_final = pd.read_csv('/Users/samuelemazzei/Desktop/Python_tesi/df_final.csv')

In [3]:
print(df_final.info())

<class 'pandas.DataFrame'>
RangeIndex: 23057 entries, 0 to 23056
Data columns (total 29 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    23057 non-null  int64  
 1   mergeid       23057 non-null  str    
 2   bmi           23057 non-null  float64
 3   depr          23057 non-null  int64  
 4   appetite      23057 non-null  int64  
 5   concen        23057 non-null  int64  
 6   fatigue       23057 non-null  int64  
 7   maxgrip       23057 non-null  float64
 8   phactiv       23057 non-null  int64  
 9   heart_attack  23057 non-null  int64  
 10  stroke        23057 non-null  int64  
 11  diabetes      23057 non-null  int64  
 12  cancer        23057 non-null  int64  
 13  drug_hbp      23057 non-null  int64  
 14  drug_sleep    23057 non-null  int64  
 15  drug_stom     23057 non-null  int64  
 16  drug_inf      23057 non-null  int64  
 17  drug_oth      23057 non-null  int64  
 18  alcohol       23057 non-null  int64  

In [4]:
tar = ['stroke','cancer','heart_attack','diabetes']
pd.DataFrame({col: df_final[col].value_counts()/ len(df_final)*100 for col in tar}).T

,0,1
stroke,96.660450,3.339550
cancer,95.727978,4.272022
heart_attack,88.107733,11.892267
diabetes,85.731014,14.268986


In [5]:
cat_tot = ['depr','appetite','concen','fatigue','phactiv',
       'drug_hbp','drug_stom','drug_sleep','drug_inf','drug_oth','alcohol','gender','astar']
num_tot = ['bmi','maxgrip','age','chol','crp','trigl','cyst_c','hdl','thb','a1c']

X_cols =  cat_tot + num_tot
y_cols = ['diabetes','stroke','heart_attack','cancer']

X = df_final[X_cols]
y = df_final[y_cols]

In [6]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

sp = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=1234)

for train_index, temp_index in sp.split(X, y):
    X_train_ub, X_temp = X.iloc[train_index], X.iloc[temp_index] #60% - 40 %
    y_train_ub, y_temp = y.iloc[train_index], y.iloc[temp_index]

sp_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=1234)
for test_index, val_index in sp_val.split(X_temp, y_temp):
    X_test, X_val = X_temp.iloc[test_index], X_temp.iloc[val_index] # 20 - 20 %
    y_test, y_val = y_temp.iloc[test_index], y_temp.iloc[val_index]

SPlitting Done - 60 training, 20 validation, 20 test, manteining the same percentage of positive.

In [7]:
from collections import Counter
print(Counter(y_train_ub['diabetes']))
print(Counter(y_train_ub['stroke']))
print(Counter(y_train_ub['heart_attack']))
print(Counter(y_train_ub['cancer']))

print(pd.DataFrame({ col : y_train_ub[col].value_counts()/len(y_train_ub) *100 for col in tar}).T)

print(Counter(y_test['diabetes']))
print(Counter(y_test['stroke']))
print(Counter(y_test['heart_attack']))
print(Counter(y_test['cancer']))

print(pd.DataFrame({ col : y_test[col].value_counts()/len(y_test) *100 for col in tar}).T)


print(Counter(y_val['diabetes']))
print(Counter(y_val['stroke']))
print(Counter(y_val['heart_attack']))
print(Counter(y_val['cancer']))
print(pd.DataFrame({ col : y_val[col].value_counts()/len(y_val) *100 for col in tar}).T)

Counter({0: 11860, 1: 1974})
Counter({0: 13372, 1: 462})
Counter({0: 12189, 1: 1645})
Counter({0: 13243, 1: 591})
                      0          1
stroke        96.660402   3.339598
cancer        95.727917   4.272083
heart_attack  88.109007  11.890993
diabetes      85.730808  14.269192
Counter({0: 3953, 1: 658})
Counter({0: 4457, 1: 154})
Counter({0: 4063, 1: 548})
Counter({0: 4414, 1: 197})
                      0          1
stroke        96.660160   3.339840
cancer        95.727608   4.272392
heart_attack  88.115376  11.884624
diabetes      85.729777  14.270223
Counter({0: 3954, 1: 658})
Counter({0: 4458, 1: 154})
Counter({0: 4063, 1: 549})
Counter({0: 4415, 1: 197})
                      0          1
stroke        96.660885   3.339115
cancer        95.728534   4.271466
heart_attack  88.096271  11.903729
diabetes      85.732871  14.267129


In [8]:
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
sme = SMOTEENN(random_state=1234,
               #smote=SMOTE(k_neighbors=5), #oversampling
               #enn=EditedNearestNeighbours(n_neighbors=3), #cleaning
               sampling_strategy=0.5
               ) #strategy = Number of minoritty after resampl / Number of majority

X_res_stroke, y_res_stroke = sme.fit_resample(X_train_ub, y_train_ub['stroke'])
y_res_stroke.name = 'stk' #resampling using SMOTEEN (remove the noise even in the majority)
X_res_diab, y_res_diab = sme.fit_resample(X_train_ub, y_train_ub['diabetes'])
y_res_diab.name = 'diab'# negatives decreasing due to noise 
X_res_ha, y_res_ha = sme.fit_resample(X_train_ub, y_train_ub['heart_attack'])
y_res_ha.name = 'ha'
X_res_can, y_res_can = sme.fit_resample(X_train_ub, y_train_ub['cancer'])
y_res_can.name = 'canc'

In [9]:
print(Counter(y_res_diab))
print(Counter(y_train_ub['diabetes']))

print(Counter(y_res_stroke))
print(Counter(y_train_ub['stroke']))

print(Counter(y_res_can))
print(Counter(y_train_ub['cancer']))

print(Counter(y_res_ha))
print(Counter(y_train_ub['heart_attack']))

Counter({0: 7118, 1: 3987})
Counter({0: 11860, 1: 1974})
Counter({0: 10147, 1: 6190})
Counter({0: 13372, 1: 462})
Counter({0: 9668, 1: 6148})
Counter({0: 13243, 1: 591})
Counter({0: 7548, 1: 4597})
Counter({0: 12189, 1: 1645})


In [10]:
def compute_scumble(y_df: pd.DataFrame) -> float:
    """
    y_df è un DataFrame con etichette 0/1 (una colonna per label).
    """
    Y = y_df.values
    n_samples, n_labels = Y.shape

    # frequenza di ogni label
    label_freq = Y.sum(axis=0)
    max_freq = label_freq.max()

    # IRLbl per ogni label
    IRLbl = max_freq / label_freq

    scumble_values = []

    for i in range(n_samples):
        active = np.where(Y[i] == 1)[0]

        if len(active) == 0:
            scumble_values.append(0)
            continue

        IR_active = IRLbl[active]

        mean_IR = IR_active.mean()
        geom_IR = np.prod(IR_active) ** (1 / len(active))

        sc_i = 1 - (geom_IR / mean_IR)
        scumble_values.append(sc_i)

    return float(np.mean(scumble_values))


sc = compute_scumble(y_train_ub).__round__(5)
print("SCUMBLE =", sc)


if sc < 0.10:
    print("Usa MLSMOTE standard")
else:
    print("SCUMBLE alto: meglio Borderline-MLSMOTE")

SCUMBLE = 0.00483
Usa MLSMOTE standard


In [11]:
#IRLbl
label_freq = y_train_ub.sum()
max_freq = label_freq.max()
IRLbl = max_freq / label_freq
print(IRLbl) #sbilanciamento tra label

#Tuning n_sample
best_sample = None
best_IR = np.inf

import sys
import mlsmote #@niteshsukhwani creator

X_sub, y_sub = mlsmote.get_minority_instace(X_train_ub, y_train_ub)

# majority subset (all rows NOT in minority subset)
minority_idx = set(mlsmote.get_index(y_train_ub))
maj_mask = ~ y_train_ub.index.isin(minority_idx)
X_maj, y_maj = X_train_ub.loc[maj_mask], y_train_ub.loc[maj_mask]

n_sample = [2000,2500,2750,2900,3500]

for i in n_sample:
    X_sub_aug, y_sub_aug = mlsmote.MLSMOTE(X_sub, y_sub,  #considering the lowest unbanceled data
                                           n_sample=i,  ## [(target = 0.5)* majority (13372)] - 462 #troppo alto
                                           random_state=1234,
                                           )
    
    X_bal = pd.concat([X_maj.reset_index(drop=True), X_sub_aug], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y_maj.reset_index(drop=True), y_sub_aug], axis=0).reset_index(drop=True)

    
    label_freq = y_bal.sum()
    IR_mean = (label_freq.max() / label_freq).mean()  # IRLbl medio
    
    print(f"n_sample={i}, IRLbl medio={IR_mean:.3f}")
    
    if IR_mean < best_IR:
        best_IR = IR_mean
        best_sample = i

print(f'The best number of sample is {best_sample}')

diabetes        1.000000
stroke          4.272727
heart_attack    1.200000
cancer          3.340102
dtype: float64
n_sample=2000, IRLbl medio=1.240
n_sample=2500, IRLbl medio=1.168
n_sample=2750, IRLbl medio=1.206
n_sample=2900, IRLbl medio=1.226
n_sample=3500, IRLbl medio=1.307
The best number of sample is 2500


** SCUMBLE INDEX ** (FINDING COOCCURANCE IN THE TRAINING SET and DECIDE WHICH MLSMOTE IS THE BEST CHOICE)

BEST CHOICE : NORMAL ML SMOTE - FOLLOWING THE PATH

In [12]:
X_sub_aug, y_sub_aug = mlsmote.MLSMOTE(X_sub, y_sub, #considering the lowest unbanceled data
                                       n_sample=best_sample, ## [(target = 0.5)* majority (13372)] - 462 #troppo alto
                                       random_state=1234)
    
X_bal = pd.concat([X_maj.reset_index(drop=True), X_sub_aug], axis=0).reset_index(drop=True)
y_bal = pd.concat([y_maj.reset_index(drop=True), y_sub_aug], axis=0).reset_index(drop=True)

print("\nTrain proportions AFTER OVERSAMPLING:")
print(y_bal.mean() * 100)


Train proportions AFTER OVERSAMPLING:
diabetes        13.291294
stroke           9.060855
heart_attack    11.840333
cancer          13.603526
dtype: float64


In [13]:
print("Train proportions AFTER recombination:\n", y_bal.mean()*100)
print(pd.DataFrame({ col : y_bal[col].value_counts()/len(y_bal) *100 for col in tar}).T)

# POST SMOTEENN (per label separato, ognuno ha il suo X)
print("\nTrain proportions AFTER SMOTEENN (per label):")
for col, y_res in zip(tar, [y_res_stroke, y_res_diab, y_res_ha, y_res_can]):
    counts = y_res.value_counts()/len(y_res)*100
    print(f"  {col}: 0: {counts[0]:.2f}   1: {counts[1]:.2f}  N={len(y_res)}")

print("Train proportions BEFORE recombination:\n", y_train_ub.mean()*100)
print(pd.DataFrame({ col : y_train_ub[col].value_counts()/len(y_train_ub) *100 for col in tar}).T)

Train proportions AFTER recombination:
 diabetes        13.291294
stroke           9.060855
heart_attack    11.840333
cancer          13.603526
dtype: float64
                    0.0        1.0
stroke        90.939145   9.060855
cancer        86.396474  13.603526
heart_attack  88.159667  11.840333
diabetes      86.708706  13.291294

Train proportions AFTER SMOTEENN (per label):
  stroke: 0: 62.11   1: 37.89  N=16337
  cancer: 0: 64.10   1: 35.90  N=11105
  heart_attack: 0: 62.15   1: 37.85  N=12145
  diabetes: 0: 61.13   1: 38.87  N=15816
Train proportions BEFORE recombination:
 diabetes        14.269192
stroke           3.339598
heart_attack    11.890993
cancer           4.272083
dtype: float64
                      0          1
stroke        96.660402   3.339598
cancer        95.727917   4.272083
heart_attack  88.109007  11.890993
diabetes      85.730808  14.269192


l denominatore (totale campioni) cresce molto, ma i nuovi campioni sintetici non necessariamente portano diabetes=1 o heart_attack=1 proporzionalmente. Quindi la loro percentuale relativa si diluisce.
È l'effetto classico del multi-label: non puoi bilanciare una label in isolamento, perché ogni campione sintetico porta con sé un intero labelset.
MLSMOTE genera campioni sintetici basandosi sulla co-occorrenza delle label. Quando crea nuovi campioni per stroke e cancer (le più sbilanciate), questi campioni sintetici hanno combinazioni di label — quindi alcuni di questi nuovi campioni avranno anche diabetes=0 o heart_attack=0.

In [14]:
# 1. PRE oversampling (y_train_ub - che hai già)
label_freq = y_train_ub.sum()
max_freq = label_freq.max()
IRLbl_pre = max_freq / label_freq
scumble_pre = compute_scumble(y_train_ub)

print(" PRE OVERSAMPLING")
print("IRLbl:\n", IRLbl_pre)
print("SCUMBLE =", round(scumble_pre, 5))

# 2. POST SMOTEENN (se hai salvato y_train_smoteenn)
smoteenn_labels = {
    'stroke':      y_res_stroke,
    'diabetes':    y_res_diab,
    'heart_attack': y_res_ha,
    'cancer':      y_res_can,
}

min_len = min(len(s) for s in smoteenn_labels.values())
y_smoteenn_proxy = pd.DataFrame(
    {col: s.values[:min_len] for col, s in smoteenn_labels.items()}
)
label_freq_sme = y_smoteenn_proxy.sum()
IRLbl_sme      = label_freq_sme.max() / label_freq_sme
scumble_sme    = compute_scumble(y_smoteenn_proxy)

print("\n=== POST SMOTEENN (proxy) ===")
print(f"N samples (min tra label): {min_len}")
print("IRLbl:\n", IRLbl_sme.round(3))
print("SCUMBLE =", round(scumble_sme, 5))

# POST MLSMOTE (y_bal finale)
label_freq_post = y_bal.sum() 
max_freq_post = label_freq_post.max()
IRLbl_post = max_freq_post / label_freq_post
scumble_post = compute_scumble(pd.DataFrame(y_bal))

print("POST MLSMOTE")
print("IRLbl:\n", IRLbl_post)
print("SCUMBLE =", round(scumble_post, 5))

 PRE OVERSAMPLING
IRLbl:
 diabetes        1.000000
stroke          4.272727
heart_attack    1.200000
cancer          3.340102
dtype: float64
SCUMBLE = 0.00483

=== POST SMOTEENN (proxy) ===
N samples (min tra label): 11105
IRLbl:
 stroke          4.162
diabetes        1.000
heart_attack    1.121
cancer          2.775
dtype: float64
SCUMBLE = 0.01886
POST MLSMOTE
IRLbl:
 diabetes        1.023491
stroke          1.501351
heart_attack    1.148914
cancer          1.000000
dtype: float64
SCUMBLE = 0.0006


In [15]:
def imbalance_rate(y: pd.DataFrame) -> pd.Series:
    return y.apply(lambda col: (col == 0).sum() / (col == 1).sum())

# 1. PRE oversampling
IR_pre = imbalance_rate(y_train_ub)
scumble_pre = compute_scumble(y_train_ub)

print ("PRE OVERSAMPLING")
print(f"N samples: {len(y_train_ub)}")
print("IR:\n", IR_pre.round(3))
print("SCUMBLE =", round(scumble_pre, 5))

# 2. POST SMOTEENN (proxy)
smoteenn_labels = {
    'stroke':       y_res_stroke,
    'diabetes':     y_res_diab,
    'heart_attack': y_res_ha,
    'cancer':       y_res_can,
}

min_len = min(len(s) for s in smoteenn_labels.values())
y_smoteenn_proxy = pd.DataFrame(
    {col: s.values[:min_len] for col, s in smoteenn_labels.items()}
)

IR_sme     = imbalance_rate(y_smoteenn_proxy)
scumble_sme = compute_scumble(y_smoteenn_proxy)

print("POST SMOTEENN")
print(f"N samples (min): {min_len}")
print("IR\n", IR_sme.round(3))
print("SCUMBLE =", round(scumble_sme, 5))

# 3. POST MLSMOTE
IR_post     = imbalance_rate(y_bal)
scumble_post = compute_scumble(y_bal)

print("POST MLSMOTE")
print(f"N samples: {len(y_bal)}")
print("IR:\n", IR_post.round(3))
print("SCUMBLE =", round(scumble_post, 5))

PRE OVERSAMPLING
N samples: 13834
IR:
 diabetes         6.008
stroke          28.944
heart_attack     7.410
cancer          22.408
dtype: float64
SCUMBLE = 0.00483
POST SMOTEENN
N samples (min): 11105
IR
 stroke          10.592
diabetes         1.785
heart_attack     2.122
cancer           6.728
dtype: float64
SCUMBLE = 0.01886
POST MLSMOTE
N samples: 16334
IR:
 diabetes         6.524
stroke          10.036
heart_attack     7.446
cancer           6.351
dtype: float64
SCUMBLE = 0.0006


In [16]:
import pickle
with open('balanced_data.pkl', 'wb') as b:
    pickle.dump({
        'X_bal': X_bal,
        'y_bal': y_bal,
        'X_test': X_test,
        'y_test': y_test,
        'X_val': X_val,
        'y_val': y_val,
        
        'X_res_can': X_res_can,
        'X_res_diab': X_res_diab,
        'X_res_stroke': X_res_stroke,
        'X_res_ha': X_res_ha,
        'y_res_can': y_res_can,
        'y_res_diab': y_res_diab,
        'y_res_stroke': y_res_stroke,
        'y_res_ha': y_res_ha,
        'x_train_ub' : X_train_ub,
        'y_train_b' : y_train_ub
    }, b)